In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import vgg16, resnet18, alexnet

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Transformations for CIFAR-10
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# Load CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# Define LeNet
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = nn.functional.relu(self.conv1(x))
        x = nn.functional.max_pool2d(x, 2)
        x = nn.functional.relu(self.conv2(x))
        x = nn.functional.max_pool2d(x, 2)
        x = x.view(-1, 16 * 5 * 5)
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Load Pretrained Models and Adjust for CIFAR-10
def get_pretrained_model(name):
    if name == "AlexNet":
        model = alexnet(pretrained=False)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)
    elif name == "VGGNet":
        model = vgg16(pretrained=False)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)
    elif name == "ResNet":
        model = resnet18(pretrained=False)
        model.fc = nn.Linear(model.fc.in_features, 10)
    return model

# Training function
def train(model, trainloader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(trainloader, 0):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {running_loss / len(trainloader):.4f}')

# Testing function
def test(model, testloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy:.2f}%')
    return accuracy

# Main function to run and compare models
def run_model_comparison():
    # Define models to compare
    models = {
        'LeNet': LeNet(),
        'AlexNet': get_pretrained_model("AlexNet"),
        'VGGNet': get_pretrained_model("VGGNet"),
        'ResNet': get_pretrained_model("ResNet")
    }

    # Initialize results dictionary
    results = {}

    # Train and evaluate each model
    for name, model in models.items():
        print(f'\nTraining {name}...')
        model = model.to(device)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()

        # Train model
        train(model, trainloader, criterion, optimizer, epochs=10)

        # Test model
        accuracy = test(model, testloader)
        results[name] = accuracy

    # Display comparison results
    print("\nModel Comparison Results:")
    for model_name, acc in results.items():
        print(f'{model_name}: {acc:.2f}%')

# Run model comparison
run_model_comparison()


Files already downloaded and verified
Files already downloaded and verified


C:\Users\opdha\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\opdha\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



Training LeNet...
Epoch [1/10], Loss: 1.8596
Epoch [2/10], Loss: 1.5948
Epoch [3/10], Loss: 1.4840
Epoch [4/10], Loss: 1.4176
Epoch [5/10], Loss: 1.3740
Epoch [6/10], Loss: 1.3340
Epoch [7/10], Loss: 1.2994
Epoch [8/10], Loss: 1.2766
Epoch [9/10], Loss: 1.2488
Epoch [10/10], Loss: 1.2276
Accuracy: 56.06%

Training AlexNet...


RuntimeError: Given input size: (256x1x1). Calculated output size: (256x0x0). Output size is too small